# Phase 0 — Data Inspect + Patient-Level Split
ROP-VL / EyeCLIP-ROP

In [1]:
from pathlib import Path
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

ROOT = Path("E:/rop/dataset/ROP-VL")
IMG_DIR = ROOT/"Image"
META_PATH = ROOT/"img_info.xlsx"
DESC_PATH = ROOT/"disease_description.xlsx"


## 1. Inspect metadata + clinical text

In [2]:

def inspect():
    """Print dataset stats. Returns meta df (n_images, 8)."""

    meta = pd.read_excel(META_PATH)

    print("Columns:", meta.columns.tolist())
    print("Shape:", meta.shape)
    print("Patients:", meta["patient_id"].nunique())

    print("Classes:")
    print(meta["categories"].value_counts())

    print("\nMissing:")
    print(meta.isnull().sum())

    print("\nClinical text: class-level only (disease_description.xlsx). No per-image text.")

    return meta


meta = inspect()

Columns: ['image', 'patient_id', 'gender', 'gestational_age', 'birth_weight', 'plurality', 'delivery_mode', 'categories']
Shape: (2020, 8)
Patients: 1116
Classes:
categories
normal               623
Laser-treated ROP    345
Stage 2 ROP          338
Stage 3 ROP          241
Stage 1 ROP          236
A-ROP                148
Stage 4 ROP           59
Stage 5 ROP           30
Name: count, dtype: int64

Missing:
image              0
patient_id         0
gender             0
gestational_age    0
birth_weight       0
plurality          0
delivery_mode      0
categories         0
dtype: int64

Clinical text: class-level only (disease_description.xlsx). No per-image text.


## 2. Check images exist + open

In [3]:
def check_images(meta):
    """Verify every image path exists and opens. Returns list of bad paths."""
    bad = []
    for _, row in meta.iterrows():
        path = IMG_DIR / row["categories"] / row["image"]
        if not path.exists():
            bad.append(str(path))
            continue
        try:
            Image.open(path).verify()
        except Exception:
            bad.append(str(path))
    print(f"Corrupt/missing: {len(bad)}")
    return bad

bad = check_images(meta)


Corrupt/missing: 0


## 3. Patient-level 70/15/15 split (no leakage)

In [4]:
def patient_level_split(meta, test_size=0.15, val_size=0.15, seed=42):
    """
    Patient-level split. Returns train_df, val_df, test_df.
    Input: meta (n, 8). Output: three dfs with same columns.
    """
    patient_class = (
        meta.groupby("patient_id")["categories"]
        .agg(lambda x: x.mode()[0])
    )
    train_ids, temp_ids = train_test_split(
        patient_class.index,
        test_size=test_size + val_size,
        stratify=patient_class,
        random_state=seed
    )
    val_ids, test_ids = train_test_split(
        temp_ids,
        test_size=0.5,
        stratify=patient_class.loc[temp_ids],
        random_state=seed
    )
    train_df = meta[meta["patient_id"].isin(train_ids)].reset_index(drop=True)
    val_df   = meta[meta["patient_id"].isin(val_ids)].reset_index(drop=True)
    test_df  = meta[meta["patient_id"].isin(test_ids)].reset_index(drop=True)
    print(f"Train: {len(train_df)} imgs / {train_df.patient_id.nunique()} pts")
    print(f"Val:   {len(val_df)} imgs / {val_df.patient_id.nunique()} pts")
    print(f"Test:  {len(test_df)} imgs / {test_df.patient_id.nunique()} pts")
    # assert no patient overlap
    assert set(train_df.patient_id) & set(val_df.patient_id) == set()
    assert set(train_df.patient_id) & set(test_df.patient_id) == set()
    assert set(val_df.patient_id) & set(test_df.patient_id) == set()
    return train_df, val_df, test_df

train_df, val_df, test_df = patient_level_split(meta)
train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)
print("Saved train/val/test.csv")


Train: 1411 imgs / 781 pts
Val:   289 imgs / 167 pts
Test:  320 imgs / 168 pts
Saved train/val/test.csv
